# Video inference

In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent.resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "output_clips"
REPORT_DIR = PROJECT_ROOT / "reports"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"

print("cwd =", Path.cwd())

### Finetune osnet

In [ ]:
from scripts.track_players_iou_v2 import run_role_inference_video

In [ ]:
game_id = "game_9_H1"

video_path = DATA_DIR / rf"{game_id}\{game_id}.mp4"
csv_path   = DATA_DIR / rf"{game_id}\players.csv"
ckpt_path = CHECKPOINTS_DIR / rf"osnet_supcon_best.pth"

out_csv_labeled   = DATA_DIR / rf"{game_id}\players_labeled.csv"
out_video_annotated  = DATA_DIR / rf"{game_id}\{game_id}_labeled.mp4"

In [ ]:
max_age=30
min_samples_per_track=10

In [ ]:
# Tracking
result = run_role_inference_video(
    video_path         = video_path,
    csv_path           = csv_path,
    model_name         = "osnet",
    ckpt_path          = ckpt_path,
    labeled_csv_path   = out_csv_labeled,
    labeled_video_path = out_video_annotated,
    device             = "cuda",
    iou_thr=0.3,
    max_age=max_age,
    max_frames=None,
    batch_size=64,
    pad=0,
    min_samples_per_track=min_samples_per_track,
    is_umap=False,
)
print(result["summary"])

### Colour Histogram 32 baseline

In [ ]:
for i in range(1, 10):
    game_id = f"game_{i}_H1"
    print(f"========================{game_id}========================")

    video_path = DATA_DIR / rf"{game_id}\{game_id}.mp4"
    csv_path   = DATA_DIR / rf"{game_id}\players.csv"

    out_csv_labeled   = DATA_DIR / rf"{game_id}\players_labeled_hist.csv"
    out_video_annotated  = DATA_DIR / rf"{game_id}\{game_id}_labeled_hist.mp4"
    max_age=30
    min_samples_per_track=10

    # Tracking
    result = run_role_inference_video(
        video_path         = video_path,
        csv_path           = csv_path,
        model_name         = "color_hist",
        labeled_csv_path   = out_csv_labeled,
        labeled_video_path = out_video_annotated,
        device             = "cpu",
        max_age=max_age,
        min_samples_per_track=min_samples_per_track,
        is_umap=False,
    )
    print(result["summary"])

### Colour Histogram 64 saturation baseline

In [ ]:
for i in range(1, 10):
    game_id = f"game_{i}_H1"
    print(f"========================{game_id}========================")

    video_path = DATA_DIR / rf"{game_id}\{game_id}.mp4"
    csv_path   = DATA_DIR / rf"{game_id}\players.csv"

    out_csv_labeled   = DATA_DIR / rf"{game_id}\players_labeled_hist_64_sat.csv"
    out_video_annotated  = DATA_DIR / rf"{game_id}\{game_id}_labeled_hist_64_sat.mp4"
    max_age=30
    min_samples_per_track=10

    # Tracking
    result = run_role_inference_video(
        video_path         = video_path,
        csv_path           = csv_path,
        model_name         = "color_hist_64_sat",
        labeled_csv_path   = out_csv_labeled,
        labeled_video_path = out_video_annotated,
        device             = "cpu",
        max_age=max_age,
        min_samples_per_track=min_samples_per_track,
        is_umap=False,
    )
    print(result["summary"])

## Metrics calculation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

from src.metrics import clustering_accuracy, macro_f1_clustering
from src.visualization import plot_confusion_matrix_clustering
from scripts.benchmark import evaluate_video_inference


### Colour Histogram 32 baseline

In [ ]:
results = evaluate_video_inference(
    game_ids=["game_1_H1", "game_2_H1", "game_3_H1", "game_4_H1", "game_5_H1",
               "game_6_H1", "game_7_H1", "game_8_H1", "game_9_H1",],
    clips_dir=DATA_DIR,
    labeled_filename= "players_labeled_hist.csv",
    players_filename= "players.csv",
)

In [ ]:
results

### Colour Histogram 64 baseline

In [ ]:
results = evaluate_video_inference(
    game_ids=["game_1_H1", "game_2_H1", "game_3_H1", "game_4_H1", "game_5_H1",
               "game_6_H1", "game_7_H1", "game_8_H1", "game_9_H1",],
    clips_dir=DATA_DIR,
    labeled_filename= "players_labeled_hist_64_sat.csv",
    players_filename= "players.csv",
)

In [ ]:
results

In [ ]:
# from src.reporter import Reporter
# REPORT_DIR = PROJECT_ROOT / "reports"

# reporter = Reporter(REPORT_DIR, experiment="evaluate_video_inference_hist_64_kmeans")
# reporter.save_table(results, "metrics_on_trakcs", fmt="csv")

### Finetune osnet

In [ ]:
game_id = "game_3_H1"
labeles = DATA_DIR / f"{game_id}/players_labeled.csv"
players = DATA_DIR / f"{game_id}/players.csv"

labeled = pd.read_csv(labeles)
players = pd.read_csv(players)

merged = labeled.merge(
    players[["frame_idx", "player_id", "role_name", "left2right"]],
    on=["frame_idx", "player_id"],
    how="inner",
)

merged = merged[
    merged["role_name"].notna() &
    merged["role_label"].notna() &
    (merged["role_label"] != "unknown")
].copy()

print(f"Strings after join: {len(merged)}")
print(f"Unique tracks: {merged['track_id'].nunique()}")

In [ ]:
def canonicalize_gt(role_name: str, left2right) -> str:
    rn = str(role_name).strip().lower()
    if "goalkeeper" in rn:
        return "goalkeeper"
    try:
        l2r = int(left2right)
    except (ValueError, TypeError):
        return "goalkeeper"
    return "team_left" if l2r == 1 else "team_right"

merged["gt_role"] = merged.apply(
    lambda r: canonicalize_gt(r["role_name"], r["left2right"]), axis=1
)

majority = lambda s: s.value_counts().idxmax()
track_df = (
    merged.groupby("track_id")
    .agg(gt_role=("gt_role", majority), role_label=("role_label", majority))
    .reset_index()
)

all_labels = sorted(
    set(track_df["gt_role"].unique()) | set(track_df["role_label"].unique())
)
le = LabelEncoder()
le.fit(all_labels)


y_true    = le.transform(track_df["gt_role"].values)
y_cluster = le.transform(track_df["role_label"].values)

print(f"lables: {dict(enumerate(le.classes_))}")
print(f"GT distribution:   { {k: int(v) for k, v in zip(le.classes_, np.bincount(y_true))} }")
print(f"Pred distribution: { {k: int(v) for k, v in zip(le.classes_, np.bincount(y_cluster))} }")

In [ ]:
acc    = clustering_accuracy(y_true, y_cluster)
macro_f1 = macro_f1_clustering(y_true, y_cluster)

print(f"  clustering_accuracy : {acc:.4f}")
print(f"  macro_f1_cluster    : {macro_f1:.4f}")


In [ ]:
y_cluster = pd.factorize(y_cluster)[0]
y_true    = pd.factorize(y_true)[0]

cm_df, fig = plot_confusion_matrix_clustering(y_true, y_cluster, normalize=False)

In [ ]:
# from src.reporter import Reporter
# REPORT_DIR = PROJECT_ROOT / "reports"

# reporter = Reporter(REPORT_DIR, experiment="evaluate_video_inference_osnet_kmeans")
# reporter.save_figure(fig, "confusion_matrix_game_3", fmt="png")

In [ ]:
results = evaluate_video_inference(
    game_ids=["game_1_H1", "game_2_H1", "game_3_H1", "game_4_H1", "game_5_H1",
               "game_6_H1", "game_7_H1", "game_8_H1", "game_9_H1",],
    clips_dir=DATA_DIR,
)

In [ ]:
results

In [ ]:
# from src.reporter import Reporter
# REPORT_DIR = PROJECT_ROOT / "reports"

# reporter = Reporter(REPORT_DIR, experiment="evaluate_video_inference_hist_32_kmeans")
# reporter.save_table(results, "metrics_on_trakcs", fmt="csv")